<div style="background:#102A43;color:#ffffff;border-radius:11px;padding:30px 28px 24px;border-top:7px solid #18B7AC;">
<div style="color:#67DED3;letter-spacing:2px;font-weight:700;font-size:12px;">INTRODUCTION TO MACHINE LEARNING · GAZA SKY GEEKS</div>
<div style="font-size:32px;font-weight:750;line-height:1.25;margin:12px 0 9px;">Lecture 07 · Data Quality:<br>Cleaning and Understanding Imperfect Data</div>
<div style="font-size:15px;color:#DDEAF3;">Instructor: Diaeddin Rimawi &nbsp; | &nbsp; 7 November 2026 &nbsp; | &nbsp; Part 2: Data Preprocessing and Proximity Measures</div>
<div style="font-size:17px;color:#ffffff;border-top:1px solid #466178;padding-top:16px;margin-top:18px;"><strong>Today's challenge:</strong> Before a Machine Learning model can learn from data, how can we decide whether the data is trustworthy and suitable for learning?</div>
</div>

**Our investigation:** Real observations → Recognize quality problems → Investigate their causes → Understand their impact → Choose justified corrections → Verify the result.

We continue with the **same Palmer Penguins dataset** from L03–L06. This time we look for problems *inside* the observations, not just patterns *between* observations.

### What you will be able to do

By the end of this lecture, you should be able to:

- Explain the difference between **data quantity, representativeness, and quality**.
- Audit a DataFrame for **missing, repeated, inconsistent, and invalid** information.
- Compare **removing and filling missing values**, and explain what each choice changes.
- Identify a **suspected outlier** without assuming it is an error.
- Apply and verify **traceable cleaning decisions** on a separate copy of the data.
- Explain why **training data**, not test data, should determine data-dependent cleaning values.

We will **not** train a model, scale features, or encode categories today; those topics come later.

### From L06 to L07: Is a representative split enough?

In L06 we discovered that **which rows we sample** can change what we conclude. Assignment 03 asks you to choose representative **training, validation, and test** splits using species, island, and body-mass summaries.

But imagine that the split is perfectly representative and:

- A training row has **no body-mass measurement**.
- One penguin is entered **twice**.
- A species name is written in **different ways**.
- A body mass recorded in **kilograms** appears in a column labeled **grams**.

**Question:** Does a good split automatically make the measurements reliable? 

**No.** We also need a data-quality investigation.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">01 · WHY QUALITY MATTERS</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Three different questions about our data</div>
<div style="font-size:14px;color:#DDEAF3;margin-top:4px;">How much data do we have, whom does it represent, and can we trust it?</div>
</div>

### Quantity, representativeness, and quality are related—not identical

| Concern | Question | Example |
|:--|:--|:--|
| **Quantity** | How many observations are available? | We have **344 rows**. |
| **Representativeness** | Do these rows reflect the situation we want to study? | Are species and islands adequately represented in a split? |
| **Quality** | Are the recorded values reliable and usable? | Is a body mass missing, mistyped, or recorded in the wrong unit? |

- More rows do not fix systematic errors or biased sampling.
- Removing every incomplete record may improve **completeness** but reduce **quantity** and **representativeness**.
- Good data quality does not prove that our future ML predictions will be correct.

**Our goal:** Make careful choices, not make the table look perfect at any cost.

### Can you spot the problem?

This is an **illustrative table, not the original scientific data**. 

A dash means a value was not recorded.

| Record | Species label | Body mass (g) | What would you check? |
|:--|:--|--:|:--|
| A | Gentoo | 5,200 | Ordinary-looking record |
| B | Gentoo | — | **Missing** measurement |
| C | Gentoo | 5,200 | May be a **duplicate** of A—or a different penguin |
| D | `gentoo ` | 5,100 | **Inconsistent** spelling and trailing space |
| E | Gentoo | −4,700 | **Invalid** negative body mass |

A separate issue could be a mass of **4.7 g** when **4.7 kg** was intended. The number is positive, but the **unit** may be wrong.


### What does “data quality” include?

| Dimension | A useful question |
|:--|:--|
| **Completeness** | Are required values present? |
| **Accuracy** | Do values reflect the real observations? |
| **Consistency** | Are names, formats, and units used consistently? |
| **Validity** | Do values satisfy known rules (e.g., mass must be positive)? |
| **Uniqueness** | Has the same real-world observation been recorded more than once? |
| **Relevance** | Are the attributes useful for our intended question? |

**Important distinction:** We can often detect a **rule violation** directly. We usually cannot confirm **accuracy** without a trusted reference, additional records, or domain knowledge. An extreme value is not automatically wrong.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">02 · DIAGNOSING THE DATA</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Audit before you repair</div>
<div style="font-size:14px;color:#DDEAF3;margin-top:4px;">What is actually missing, suspicious, or inconsistent in our original file?</div>
</div>

### Begin with the untouched Palmer Penguins observations

Our dataset describes three penguin species using eight attributes. The numeric measurements include **bill length (mm)**, **bill depth (mm)**, **flipper length (mm)**, and **body mass (g)**.

- Use the bundled `data/penguins.csv`, exactly as in earlier lectures.
- Store it in `penguins` and **do not overwrite** it.
- A **data-quality audit** records what we can observe before suggesting any fixes.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
NAVY = "#102A43"
TEAL = "#158B94"
SPECIES_ORDER = ["Adelie", "Chinstrap", "Gentoo"]
SPECIES_COLORS = {"Adelie": "#158B94", "Chinstrap": "#DB9137", "Gentoo": "#557AB5"}

path = Path("data/penguins.csv")
penguins = pd.read_csv(path)
print("Dataset shape:", penguins.shape)
display(penguins.head())

### A first look: types, categories, and numerical ranges

We already know these Pandas operations from L03–L05. Here their purpose is **quality checking**:

- `.info()` reveals types and non-missing counts.
- `.unique()` / `.value_counts()` can expose unexpected category labels.
- `.describe()` gives minimum and maximum values that may deserve investigation.
- `.duplicated()` flags rows matching an earlier row in all columns.

In [ ]:
penguins.info()
print("\nSpecies:", penguins["species"].unique().tolist())
print("Islands:", penguins["island"].unique().tolist())
print("Sex values:", penguins["sex"].unique().tolist())
print("Exact duplicate rows:", penguins.duplicated().sum())
display(penguins.describe().loc[["min", "max"]].round(1))

### Measure missingness, not just its presence

A useful audit reports both:

- **Count:** how many cells in each column are missing?
- **Percentage:** what share of the **344 rows** is missing that attribute?

For a column $j$, the missing-value percentage is:

$$
\text{Missing percentage}_j = \frac{\text{missing cells in column }j}{\text{number of rows}}\times100.
$$

Percentages let us compare attributes. They do **not** reveal why the values are missing.

In [ ]:
missing_counts = penguins.isna().sum()
missing_audit = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": (100 * missing_counts / len(penguins)).round(2)
})
display(missing_audit)
print("Rows with at least one missing field:", penguins.isna().any(axis=1).sum())
print("Total missing cells:", int(missing_counts.sum()))

**What should you notice?**

- There are **2 missing values in each of the four numerical measurement columns**.
- There are **11 missing `sex` values**; these include both rows with missing measurements.
- **11 rows**, not 19 rows, have at least one missing value. **19** is the number of missing *cells*.
- The original file has **no exact duplicate rows**. That does **not** prove it has no repeated real-world observations.

We have identified issues; we have **not corrected** anything yet.

In [ ]:
missing_plot = missing_audit.loc[missing_audit["missing_count"] > 0].sort_values("missing_count")
fig, ax = plt.subplots(figsize=(8.5, 3.6))
ax.barh(missing_plot.index, missing_plot["missing_percent"], color=TEAL)
for y, percent in enumerate(missing_plot["missing_percent"]):
    ax.text(percent + 0.12, y, f"{percent:.2f}%", va="center", fontsize=10)
ax.set(title="Missing values in Palmer Penguins", xlabel="Missing observations (%)", ylabel="Attribute")
ax.set_xlim(0, missing_plot["missing_percent"].max() + 1.0)
plt.tight_layout()
plt.show()

### Look for suspect values with explicit rules

- **Category rule:** `species` should use one of the three known labels.
- **Physical rule:** a recorded body mass must be **greater than zero**.
- **Domain screening rule:** values well outside a plausible range deserve review; the **chosen range must be justified**, not guessed from a box plot.

Check the first two rules on the *original* data. Missing masses are **unknown**, not automatically invalid measurements.

In [ ]:
unknown_species = penguins["species"].notna() & ~penguins["species"].isin(SPECIES_ORDER)
nonpositive_mass = penguins["body_mass_g"].notna() & (penguins["body_mass_g"] <= 0)
print("Unknown species labels:", unknown_species.sum())
print("Non-positive recorded masses:", nonpositive_mass.sum())
print("Exact duplicates:", penguins.duplicated().sum())
print("Body-mass range (g):", penguins["body_mass_g"].min(), "to", penguins["body_mass_g"].max())

### The first audit report

A short audit should answer **what**, **how often**, and **how certain**:

| Finding in the original file | Status | What next? |
|:--|:--|:--|
| Missing numerical and `sex` values | **Confirmed absence** | Check extent and possible consequences |
| Exact duplicate rows | **None found** | Do not invent a duplicate problem |
| Unexpected species names | **None found** | Keep this rule in future checks |
| Extreme mass values | **Not yet evaluated** | Compare distributions and species later |

**Detection is not correction.** Even a confirmed missing value does not immediately tell us whether to remove or fill it.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">03 · MISSING INFORMATION</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">When values are absent, what should we do?</div>
<div style="font-size:14px;color:#DDEAF3;margin-top:4px;">How does the choice to keep, remove, or fill values change what we can learn?</div>
</div>

### Missing is not the same as zero

- **`NaN` / `None`:** Common ways to represent information that is not available in Pandas.
- **Empty text `""` or placeholders `"?"`:** These may look like missing data but **are not always detected** by `.isna()` until we explicitly convert them.
- **Zero `0`:** A real numerical value. A body mass of zero is physically invalid, **not a suitable replacement for a missing mass**.
- **“Not applicable”:** Sometimes an attribute does not apply; replacing it with a typical value would change its meaning.

**Rule:** Understand what the source uses to represent absence. Never silently turn all missing values into zero.

In [ ]:
example_entries = pd.Series([np.nan, None, "", "?", "0"], name="raw_value")
display(pd.DataFrame({"value": example_entries, "is_missing_by_pandas": example_entries.isna()}))
standardized_entries = example_entries.replace({"": np.nan, "?": np.nan})
print("Missing before standardization:", example_entries.isna().sum())
print("Missing after standardization:", standardized_entries.isna().sum())
print("The text '0' is not missing:", pd.notna(standardized_entries.iloc[-1]))

### Why might a value be missing?

- A measurement was **not collected** or the equipment failed.
- A form was **left unanswered** or a record was incomplete.
- The attribute was **not available or not applicable** in that situation.
- The missingness could be **related to a group, time, or measurement difficulty**.

**Important:** Missing values may **not** occur randomly. If one group loses more records than another, removing incomplete rows can alter the dataset's composition. We can inspect this pattern but cannot infer its cause from the CSV alone.

In [ ]:
missing_sex_by_species = pd.crosstab(
    penguins["species"], penguins["sex"].isna(),
    rownames=["Species"], colnames=["Sex missing?"]
).reindex(SPECIES_ORDER)
display(missing_sex_by_species)
print("Observed sex counts:")
display(penguins.groupby("species")["sex"].count().reindex(SPECIES_ORDER).rename("known_sex_count"))

**Interpretation:** The missing `sex` values are present in **Adelie** and **Gentoo** records, but not in **Chinstrap** records in this file. We do **not** know why. Removing incomplete rows therefore changes group sizes slightly; it is not a neutral formatting step.

### Four reasonable responses to a missing value

| Choice | Benefit | Important cost or risk |
|:--|:--|:--|
| **Keep and report** | Preserves observations | Later tools may not accept missing entries |
| **Remove selected rows** | Uses only recorded values for chosen fields | Loses observations and may change group composition |
| **Remove an attribute** | Avoids a highly incomplete or irrelevant column | May lose useful information |
| **Fill using a stated rule** | Preserves row count | Inserts estimated values that may change spread and relationships |

**No strategy is automatically best.** First decide which attributes are essential to the current question.

### Compare two ways to remove incomplete rows

- `dropna()` without arguments removes a row if **any column** is missing.
- `dropna(subset=["body_mass_g"])` removes only rows without **body mass**, preserving records whose other fields are incomplete.
- `dropna(axis=1)` would remove *columns* with missing values; do not use it automatically—it would discard important measurements here.

Always use a **copy or a new variable**, and report the number of removed observations.

In [ ]:
all_complete = penguins.dropna().copy()
only_mass_complete = penguins.dropna(subset=["body_mass_g"]).copy()
print("Original rows:", len(penguins))
print("Rows after dropna() on all columns:", len(all_complete))
print("Rows after requiring body mass only:", len(only_mass_complete))
print("Rows retained with only mass required:", len(only_mass_complete) - len(all_complete))
display(pd.DataFrame({
    "original": penguins["species"].value_counts(),
    "all_columns_complete": all_complete["species"].value_counts(),
    "body_mass_present": only_mass_complete["species"].value_counts()
}).reindex(SPECIES_ORDER))

**Notice:** Requiring complete values for all eight attributes leaves **333 rows**. Requiring only body mass leaves **342 rows**. The nine-row difference matters if those other attributes are not needed for our analysis.

### What does “imputation” mean?

**Imputation** means placing a value in a missing position using an explicit rule.

- **Mean filling:** Replace missing numerical entries with the mean of recorded values.
- **Median filling:** Replace them with the median, often less affected by extreme values.
- **Categorical filling:** Sometimes a known label such as `"Unknown"` is more honest than guessing the most common category—provided that an unknown category is appropriate.

Imputed values are **estimates or markers**, not newly measured facts. Retain the reason and method in your documentation.

In [ ]:
observed_mass = penguins["body_mass_g"]
mass_mean = observed_mass.mean()
mass_median = observed_mass.median()
mean_filled = penguins.copy()
median_filled = penguins.copy()
mean_filled["body_mass_g"] = mean_filled["body_mass_g"].fillna(mass_mean)
median_filled["body_mass_g"] = median_filled["body_mass_g"].fillna(mass_median)
print(f"Observed mean: {mass_mean:.1f} g; median: {mass_median:.1f} g")
print("Missing mass values after median filling:", median_filled["body_mass_g"].isna().sum())

### Does filling change the statistics?

**Why look beyond the count?** Filling creates additional numerical entries. Even if mean filling preserves the **overall mean**, it commonly reduces the **sample standard deviation** because inserted values sit at the center.

In our real dataset only **two masses** are missing, so the change is small. That is an observation about **this dataset**, not proof that mean filling is always safe. Let's calculate it instead of assuming.

In [ ]:
strategies = {
    "Observed masses only": observed_mass.dropna(),
    "Drop any incomplete row": all_complete["body_mass_g"],
    "Require mass only": only_mass_complete["body_mass_g"],
    "Fill mass with mean": mean_filled["body_mass_g"],
    "Fill mass with median": median_filled["body_mass_g"]
}
comparison = pd.DataFrame({name: {
    "n_masses": values.count(),
    "mean_g": values.mean(),
    "median_g": values.median(),
    "std_g": values.std()
} for name, values in strategies.items()}).T
comparison["n_masses"] = comparison["n_masses"].astype(int)
display(comparison.round(2))

### A small example makes the effect easier to see

Consider **illustrative waiting times**, not penguin data: `4, 5, 6, missing, 40`. The value 40 is intentionally unusual.

- The **mean** of known values is **13.75**; the **median** is **5.5**.
- Mean filling inserts **13.75**, while median filling inserts **5.5**.
- Neither filled value was actually observed in the missing position.

**Question:** Which method better reflects a typical waiting time? Which better preserves variation? We should test, not assume.

In [ ]:
toy = pd.Series([4, 5, 6, np.nan, 40], dtype=float)
toy_strategies = {
    "Recorded only": toy.dropna(),
    "Mean fill": toy.fillna(toy.mean()),
    "Median fill": toy.fillna(toy.median())
}
toy_summary = pd.DataFrame({name: {"n": s.count(), "mean": s.mean(), "median": s.median(), "std": s.std()}
                            for name, s in toy_strategies.items()}).T
display(toy_summary.round(2))
fig, ax = plt.subplots(figsize=(8.2, 3.5))
for i, (name, values) in enumerate(toy_strategies.items()):
    ax.scatter(values, np.full(len(values), i), s=95, alpha=0.85, label=name)
ax.set(yticks=range(3), yticklabels=list(toy_strategies),
       xlabel="Illustrative waiting time (minutes)", ylabel="Strategy",
       title="Filling one missing value creates a new data point")
ax.set_xlim(0, 43)
plt.tight_layout()
plt.show()

**Interpretation:** Filling moves us from **four observed waiting times** to **five values**. The added point affects the mean or the median and generally changes variability. If the unusual 40-minute delay is genuine, automatically deleting it would throw away information.

For penguin masses, the median is less sensitive to extremes, but **neither method reconstructs the missing measurement**. Choosing a value is a compromise guided by our task.


<div style="border-left:5px solid #18B7AC;background:#EEF8F7;padding:12px 16px;border-radius:5px;"><strong style="color:#102A43;">ACTIVITY 1 — Same measurements, same penguin?</strong></div>

**Your challenge:** We have 344 penguin observations. But how many share the same measurements?

Using the original `penguins` DataFrame:

1. Count how many rows have the **same species and body mass** as an earlier row, using `duplicated(subset=[...])`.
2. Count how many rows are **exact duplicates across all columns**, using `duplicated()`.
3. Compare the two numbers. **Are you surprised?**

*Hint: `.duplicated()` marks repeated entries after the first occurrence. Use `.sum()` to count them.*

**Think about it:** If two penguins have the same species and body mass, does that mean they are the same penguin?


In [ ]:
# Activity 1 — Use copies; do not modify 'penguins'.
# TODO1

# TODO2

# TODO3

<details>
<summary><strong>Suggested answer — click to expand after trying</strong></summary>

**Suggested code**

```python
same_measurements = penguins.duplicated(
    subset=["species", "body_mass_g"]
).sum()

exact_duplicates = penguins.duplicated().sum()

print("Repeated species and body mass:", same_measurements)
print("Exact duplicate rows:", exact_duplicates)
```

**Expected results:**

- **206 rows** repeat a species–body-mass combination seen earlier.
- **0 rows** are exact duplicates across all eight attributes.

**What does this tell us?**

- Many penguins share the same species and body mass.
- This is not necessarily a data-quality problem. Different penguins can have identical measurements.
- Repeated values are not sufficient evidence that an observation was recorded twice.

**A question for our next investigation:**

If repeated measurements are normal, **how can we distinguish legitimate observations from records that were accidentally duplicated?**

And if we find duplicate-looking records, should we automatically delete them?

</details>


<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">04 · REPEATED AND INCONSISTENT RECORDS</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">When two entries look the same, are they the same?</div>
<div style="font-size:14px;color:#DDEAF3;margin-top:4px;">How can repeated records, category variants, and wrong units distort a dataset?</div>
</div>

### Identical rows are not always duplicate observations

- An **exact duplicate row** has identical values in every attribute, matching an earlier row.
- A **duplicate observation** means the *same real-world event or penguin* was recorded more than once.
- Two different penguins may coincidentally have identical recorded attributes.
- Conversely, the same penguin recorded twice might have slightly different values.

`duplicated()` detects identical rows; it does **not** prove shared identity. We need a unique observation ID or trusted provenance before removing real records with confidence.

Our original file has **zero exact duplicate rows**. We will create one in a **separate teaching copy**.

In [ ]:
duplicate_demo = pd.concat([penguins, penguins.iloc[[7]]], ignore_index=True)
print("Original rows:", len(penguins))
print("Teaching copy rows:", len(duplicate_demo))
print("Exact duplicates in teaching copy:", duplicate_demo.duplicated().sum())
display(duplicate_demo.loc[duplicate_demo.duplicated(keep=False)].tail(2))
unique_demo = duplicate_demo.drop_duplicates().copy()
print("After removing the deliberately repeated row:", len(unique_demo))

**What changed?** The added row makes that penguin's **species**, **island**, and measurements appear one extra time in frequency counts and means. We can remove it here because **we ourselves inserted a copy of row 7**. In an unfamiliar real dataset, verify identity before applying `drop_duplicates()`.

### One category, several spellings

A category value is **text**, so `"Gentoo"`, `"gentoo"`, `"Gentoo "`, and `"GENTOO"` are four distinct strings. Without cleaning, Pandas counts them as different groups.

This is a **deliberately constructed text example**, not an issue observed in the original penguin CSV.

In [ ]:
label_demo = pd.Series(["Gentoo", "gentoo", "Gentoo ", "GENTOO", "Adelie"], name="species")
print("Before standardization:")
display(label_demo.value_counts())
normalized_labels = label_demo.str.strip().str.casefold()
print("normalized_labels:", normalized_labels.to_list())
label_map = {"adelie": "Adelie", "chinstrap": "Chinstrap", "gentoo": "Gentoo"}
standard_labels = normalized_labels.map(label_map)
print("After checking known labels:")
display(standard_labels.value_counts())

**Why this cleaning works:** `.str.strip()` removes extra spaces and `.str.casefold()` gives a case-insensitive comparison. The dictionary then restores **known, canonical names**.

- Do **not** map unknown labels blindly: an unexpected species should remain visible for investigation.
- Do **not** treat this as categorical encoding. We are standardizing the *meaning of the original labels*, not converting labels to numerical ML features.

### A number can also be written in the wrong way

- `"3800 g"` is **text**, not a numerical measurement until parsed.
- A positive value of **3.65** in a `body_mass_g` column may mean **3.65 kg** was recorded as grams; it needs an evidence-based unit check.
- A negative body mass clearly violates the **physical positivity rule**.
- `pd.to_numeric(..., errors="coerce")` turns unparseable text into `NaN`. **Do not silently accept this loss of information.**

In [ ]:
# Controlled miniature examples; these are NOT rows from the original CSV.
raw_masses = pd.Series(["3800 g", "3650", "?"], name="raw_body_mass")
without_units = raw_masses.str.replace(" g", "", regex=False).replace({"?": np.nan})
parsed_masses = pd.to_numeric(without_units, errors="coerce")
display(pd.DataFrame({"raw_entry": raw_masses, "parsed_g": parsed_masses}))
print("Unparsed or unknown values:", parsed_masses.isna().sum())

# A *documented* unit mismatch would justify this, but a small number alone would not:
recorded_in_kg = 3.65
print("If confirmed as kilograms:", recorded_in_kg * 1000, "g")

### Check validity before changing anything

A validation rule can ask:

1. **Known labels:** Is the species in our accepted list?
2. **Physical requirement:** Is a recorded body mass positive?
3. **Screening range:** Does a value look implausible for this particular dataset and unit?

<!-- For our **teaching copy only**, we will screen masses outside **1,000–10,000 g**. These are broad, *illustrative review limits*.  -->

Being flagged means **investigate**, unless a trusted source confirms an error.

### A controlled challenge: a few planted problems

The next code cell constructs `dirty_activity`. Starting from the real 344 rows, it deliberately:

- Changes **two Adelie labels** to different spellings.
- Changes one mass to a **negative value** and another to a **kilogram-sized number in a gram column**.
- Appends an **exact copy of row 7**.

We know precisely which errors were planted. **No original CSV is changed.**

In [ ]:
dirty_activity = penguins.copy()
dirty_activity.loc[0, "species"] = "adelie "
dirty_activity.loc[1, "species"] = "ADELIE"
dirty_activity.loc[4, "body_mass_g"] = -3450.0          # PLANTED: negative mass
dirty_activity.loc[5, "body_mass_g"] = 3.65             # PLANTED: kg-sized entry stored in g column
dirty_activity = pd.concat([dirty_activity, penguins.iloc[[7]]], ignore_index=True)
print("Teaching-copy size:", dirty_activity.shape)
display(dirty_activity.loc[[0, 1, 4, 5, len(dirty_activity)-1]])

<div style="border-left:5px solid #18B7AC;background:#EEF8F7;padding:12px 16px;border-radius:5px;"><strong style="color:#102A43;">ACTIVITY 2 — Find and correct the planted errors</strong></div>

**Your task:** Using only `dirty_activity`, find unexpected species names, exact duplicate rows, and implausible masses. Correct the **known planted errors** using `penguins` as a trusted copy of the original measurements. Verify that the restored data match `penguins`. 

*Edit the following code cell, and explain your decision in your own words before opening the suggested answer.*

In [ ]:
# Activity 2 — Start from the planted-error teaching copy.
# TODO: identify unknown species, duplicated rows, and masses outside 1000–10000 g.
# TODO: create a repaired copy (do not change dirty_activity).
# TODO: validate the repaired copy against 'penguins'.

<details><summary><strong>Suggested answer — click to expand after trying</strong></summary>

**Suggested code**

```python
activity2_unknown = ~dirty_activity["species"].isin(SPECIES_ORDER)
activity2_bad_mass = (dirty_activity["body_mass_g"].notna()
                      & ~dirty_activity["body_mass_g"].between(1000, 10000))
print("Unknown labels:", dirty_activity.loc[activity3_unknown, "species"].tolist())
print("Repeated rows:", dirty_activity.duplicated().sum())
print("Suspicious mass row indices:", dirty_activity.index[activity3_bad_mass].tolist())
activity2_fixed = dirty_activity.copy()
activity2_fixed["species"] = activity2_fixed["species"].replace({"adelie ": "Adelie", "ADELIE": "Adelie"})
activity2_fixed.loc[[4, 5], "body_mass_g"] = penguins.loc[[4, 5], "body_mass_g"]
activity2_fixed = activity2_fixed.drop_duplicates().reset_index(drop=True)
print("Restored original:", activity2_fixed.equals(penguins))
```

**Interpretation:** Two labels, two recorded masses, and one inserted row need correction. This case can be completely restored **only because we know what we intentionally changed and still have the original source**. For an unexplained 3.65 or a repeated row in real records, the safe response is to flag it, consult provenance, and document the unresolved uncertainty—not invent a corrected value.

</details>

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">05 · NOISE AND OUTLIERS</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Is an unusual penguin a bad observation?</div>
<div style="font-size:14px;color:#DDEAF3;margin-top:4px;">Can a valid measurement be rare, and how do groups affect our judgement?</div>
</div>

### Noise, error, and an unusual value are different ideas

- **Noise:** Unwanted variation in a measurement or recording process; e.g., small fluctuations caused by measurement conditions.
- **Unusual observation:** A genuine value far from most others; it may be scientifically interesting.
- **Recorded error:** A value written or measured incorrectly, which requires evidence to confirm.
- **Invalid value:** A known rule is violated (for example, negative physical mass).

**Key principle:** An **outlier** is a reason to investigate, **not an instruction to delete**. Box plots and numerical rules identify candidates

### Recall the 1.5 × IQR rule from L05

The **first quartile** $Q_1$ marks the 25th percentile, and the **third quartile** $Q_3$ marks the 75th percentile.

$$
IQR = Q_3 - Q_1
$$

$$
\text{Lower fence} = Q_1 - 1.5\,IQR
$$

$$
\text{Upper fence} = Q_3 + 1.5\,IQR
$$

Values **outside the fences** are *flagged*. The rule is a conventional descriptive screen, not a physical law. It can behave differently when we combine groups with different typical body masses.

In [ ]:
mass = penguins["body_mass_g"].dropna()
q1_global, q3_global = mass.quantile([0.25, 0.75])
iqr_global = q3_global - q1_global
lower_global = q1_global - 1.5 * iqr_global
upper_global = q3_global + 1.5 * iqr_global
global_flags = mass[(mass < lower_global) | (mass > upper_global)]
print(f"Pooled fences: {lower_global:.1f} to {upper_global:.1f} g")
print("Pooled IQR-flagged masses:", len(global_flags))

### The same rule **within each species**

Gentoo penguins are generally heavier than Adelie or Chinstrap penguins in this dataset. A single overall fence may hide what is unusual *inside* one group.

- First inspect **within-species** summaries.
- Apply IQR fences separately, using each species' own quartiles.
- Compare flags with the actual observation and its context.

We are not claiming a species-specific IQR flag is a confirmed measurement error.

In [ ]:
group_outliers = []
group_fences = []
for species in SPECIES_ORDER:
    sample = penguins.loc[penguins["species"] == species, "body_mass_g"].dropna()
    q1, q3 = sample.quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = sample[(sample < low) | (sample > high)]
    group_fences.append({"species": species, "lower_g": low, "upper_g": high, "flagged_n": len(flagged)})
    for row_index, value in flagged.items():
        group_outliers.append({"row_index": row_index, "species": species, "body_mass_g": value})
display(pd.DataFrame(group_fences).set_index("species").round(1))
display(pd.DataFrame(group_outliers))

**What is interesting?** The overall IQR rule flags **zero** recorded body masses, while the within-species rule flags **two Chinstrap values** (**2,700 g** and **4,800 g**). Neither value is automatically impossible. We need to know more before excluding either.

Let's visualize those comparisons in a familiar **box plot**.

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.4))
mass_by_species = [penguins.loc[penguins["species"] == s, "body_mass_g"].dropna()
                   for s in SPECIES_ORDER]
box = ax.boxplot(mass_by_species, tick_labels=SPECIES_ORDER, patch_artist=True, widths=0.55)
for patch, species in zip(box["boxes"], SPECIES_ORDER):
    patch.set_facecolor(SPECIES_COLORS[species])
    patch.set_alpha(0.7)
for median_line in box["medians"]:
    median_line.set_color(NAVY)
    median_line.set_linewidth(2)
if group_outliers:
    flag_frame = pd.DataFrame(group_outliers)
    for point in flag_frame.itertuples():
        x = SPECIES_ORDER.index(point.species) + 1
        ax.scatter(x, point.body_mass_g, s=110, color="#B73544", edgecolor="white",
                   linewidth=1.5, zorder=5)
ax.set(title="Body mass by species: investigate IQR-flagged observations",
       xlabel="Penguin species", ylabel="Body mass (g)")
plt.tight_layout()
plt.show()

### Why not remove every flagged value?

- The **2,700 g** and **4,800 g** Chinstrap observations are unusual *relative to the Chinstrap group*, but may be real.
- Deleting them changes group **range**, **standard deviation**, and possibly **mean**—even if no recording error exists.
- A change in a statistic is not proof that cleaning was beneficial.

A defensible approach is to **retain and annotate** unresolved observations or investigate their provenance. Automatic deletion should not replace evidence.

In [ ]:
chinstrap_mass = penguins.loc[penguins["species"] == "Chinstrap", "body_mass_g"].dropna()
chinstrap_q1, chinstrap_q3 = chinstrap_mass.quantile([0.25, 0.75])
chinstrap_iqr = chinstrap_q3 - chinstrap_q1
chinstrap_in_fence = chinstrap_mass.between(chinstrap_q1 - 1.5 * chinstrap_iqr,
                                            chinstrap_q3 + 1.5 * chinstrap_iqr)
compare_chinstrap = pd.DataFrame({
    "All recorded Chinstrap masses": chinstrap_mass.agg(["count", "mean", "std", "min", "max"]),
    "Hypothetical: omit IQR flags": chinstrap_mass[chinstrap_in_fence].agg(["count", "mean", "std", "min", "max"])
}).T
compare_chinstrap["count"] = compare_chinstrap["count"].astype(int)
display(compare_chinstrap.round(1))

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">06 · RESPONSIBLE CLEANING</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Can we justify and verify every change?</div>
<div style="font-size:14px;color:#DDEAF3;margin-top:4px;">Build a simple, reproducible audit → decision → correction → recheck workflow.</div>
</div>

### A practical workflow for trustworthy cleaning

1. **Preserve:** Keep the original file and work on a copy.
2. **Audit:** Count missing data, duplicate rows, invalid categories, and implausible measurements.
3. **Classify:** Decide whether a problem is **confirmed**, **suspected**, or **unresolved**.
4. **Choose:** Specify an evidence-based action and record the reason.
5. **Correct:** Apply the action only where justified.
6. **Verify:** Run the same checks again and compare key statistics.
7. **Report:** State what changed and what uncertainty remains.

In our controlled case, the original CSV can act as a **trusted reference**. Real cleaning often lacks that luxury.

### Keep a decision log, not just a clean-looking table

| Identified issue | Affected entries | Documented action | Reason |
|:--|--:|:--|:--|
| Planted Adelie label variants | 2 | Restore known label | Original labels are available |
| Planted mass errors | 2 | Restore known measurements | Original recorded values are available |
| Planted repeated row | 1 | Remove inserted copy | We know we appended that row |
| Real absent measurements / sex | 19 cells | **Keep and report** for now | Missing source measurements are unknown |
| Real Chinstrap IQR flags | 2 observations | **Retain; investigate** | Unusual does not mean incorrect |

**Important:** The real missing-cell count is **19**, spread across **11 rows**. We have not solved missingness merely by correcting the planted problems.

### A crucial ML connection: when should we calculate a filling value?

In L06, Assignment 03 introduces `train`, `val`, and `test` labels. Their purposes are different:

- **Training:** Learn from the training observations (including a median used to fill missing features).
- **Validation:** Check and compare later modeling choices, without letting its values determine the training median.
- **Test:** Keep evaluation records independent of such decisions.

A fixed rule from trusted documentation—e.g., correcting a known spelling—is different from a rule **estimated from the data**. Using validation or test measurements to calculate the training median creates **data leakage**.

In [ ]:
# Tiny ILLUSTRATIVE example, not the results of Assignment 03.
train_masses = pd.Series([3500, 3700, np.nan], dtype=float)
validation_masses = pd.Series([5000, np.nan], dtype=float)
train_median = train_masses.median()
leaky_median = pd.concat([train_masses, validation_masses]).median()
print("Training-only median:", train_median, "g")
print("Median if validation values leak in:", leaky_median, "g")
print("Validation missing entry, filled with training median:",
      validation_masses.fillna(train_median).tolist())

**What should you notice?** Using validation values changes this example's filling rule from **3,600 g** to **3,700 g**. That means information from the future evaluation data has influenced preparation. The difference is small here, but the principle matters.

**For now:** Understand why the training/validation/test distinction affects cleaning decisions. We will implement complete leakage-safe preprocessing pipelines in **L10**.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">07 · SYNTHESIS AND THE NEXT QUESTION</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">What makes a cleaning decision responsible?</div>
<div style="font-size:14px;color:#DDEAF3;margin-top:4px;">Can we trust the dataset we are preparing for Machine Learning?</div>
</div>

### Our investigation, in seven decisions

| Question | What we learned |
|:--|:--|
| Is information missing? | Count missing **cells and affected rows**; investigate reasons. |
| Should we remove incomplete rows? | Decide based on **required attributes** and possible lost information. |
| Should we fill missing entries? | Explain the assumed rule and measure its effect on statistics. |
| Are rows duplicated? | Identical values **do not prove** identical real-world observations. |
| Are names and units consistent? | Repair known mistakes; **flag unknowns instead of guessing**. |
| Is a numerical observation unusual? | **IQR flags require investigation**, often within meaningful groups. |
| Can we reproduce the result? | Keep copies, a decision log, and a **before-and-after audit**. |


### Three final discussion questions

1. **Completeness vs. representativeness:** Could dropping every row without a `sex` value affect comparisons among species? What evidence would help us decide?
2. **Validity vs. unusualness:** Is an unusual Chinstrap mass necessarily wrong? What would you ask the researchers before changing it?
3. **Cleaning vs. feature engineering:** If our labels and masses are now trustworthy, are these attributes automatically in the best form for a Machine Learning algorithm?

Takeaway: **Representative sampling and careful data cleaning support reliable analysis, but neither guarantees reliable model evaluation.**

### Looking ahead — Lecture 08: Feature Engineering

Today we asked: **Can we trust and clean our observations?**

In **L08 — Feature Engineering: Transforming Data for Learning**, we will ask a different question:

> *Now that we can recognize and correct problems, how can we turn valid measurements and categories into useful representations that Machine Learning algorithms can learn from?*

Our progression continues:

- **L07:** Investigate validity, consistency, and missingness.
- **L08:** Transform useful attributes into meaningful features.
- **L09:** Measure similarity and distance between observations.
- **L10:** Select features and build reliable preprocessing pipelines.

**Remember:** Good Machine Learning does not begin with choosing an algorithm. It begins with understanding the data we are asking it to learn from.

### Sources, acknowledgements, and further reading

- **Palmer Penguins:** Horst, A. M., Hill, A. P., & Gorman, K. B. (2020), *palmerpenguins: Palmer Archipelago (Antarctica) penguin data* ([documentation](https://allisonhorst.github.io/palmerpenguins/); [Zenodo DOI 10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218)); original ecological observations: Gorman, K. B., Williams, T. D., & Fraser, W. R. (2014), *PLoS ONE*. Our bundled 344-row dataset is the same local file used in earlier lectures.
- **Teaching inspirations:** Selected concepts were adapted from the teaching material of **Prof. Antonio Liotta**.
- **Pandas reference:** [`DataFrame.isna`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html), [`DataFrame.dropna`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html), [`Series.fillna`](https://pandas.pydata.org/docs/reference/api/pandas.Series.fillna.html), [`DataFrame.duplicated`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html), [`DataFrame.drop_duplicates`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html).
